# TP03: Language Processing with `PLY` and `Lark`

## Grammar Engineering

## MEI/2025-26

#### Nuno Macedo
Universidade do Minho

The following programming exercises are intended to be solved with two parser generator modules for Python, [PLY](https://ply.readthedocs.io/) and [Lark](https://lark-parser.readthedocs.io/).

# Module [`Ply`](https://ply.readthedocs.io/) basics

- A Python implementation of the popular lex/yacc compiler front-end tools

- The scanning and parsing phases are distinct:
  - The lexer definition is given as a set of regular expressions that represent different token types
  - The parser definition is given as a grammar in BNF whose terminal symbols are token types

- It relies on a *translating grammar*, a grammar enhanced with semantic actions that drive a *syntax-directed translation*

- This makes it easier to write simple direct translations, makes the translation tightly coupled to the grammar
  - Less suitable for multiple translations
  - Harder to understand the semantics
  - Less prone to formal analysis

- Supports LALR(1) parsing (Look-Ahead Left-to-right Rightmost derivation with 1 token of lookahead)

- In traditional lex/yacc, the specifications are given as separate files and compiled into a target language
  - In PLY they are written directly in Python (uses reflection)

- *Note:* since PLY relies on reflection, it is not prone to having different specifications in the same module
  - This leads to some conflicts when writing specifications in notebooks
  - Either run outside the notebook or restart the session for each exercise


In [ ]:
!pip install Ply

## [`ply.lex`](https://ply.readthedocs.io/en/latest/ply.html#lex) basics

- The list of token types is given in a variable `tokens`

- After scanning, each token has a *type* and a *value* assigned (by default, the lexeme)

- Each token type `T` is defined by a variable `t_T` that is assigned a regular expression
  - Either directly a string with a RE
  - Or a function that has a RE as docstring and further actions

- Can also define single-symbol literal tokens in variable *literals* (both type and value become the symbol)

- The master RE is built in the following order:
  - Function-defined tokens by order they appear
  - Literal-defined tokens by descending length

- Other special function:
  - `t_ignore` to automatically ignore certain symbols
  - `t_error` to handle errors

## Example: Simple arithmetic expressions

In [ ]:
import ply.lex as lex
import math

tokens = (
  'INT', 'FLOAT', 'PLUS', 'MINUS', 'DIVIDE', 'TIMES'
)

literals = ('(', ')')

t_ignore = ' \t\n'
t_PLUS    = r'\+'
t_MINUS   = r'-'
t_TIMES   = r'\*'
t_DIVIDE  = r'/'

def t_FLOAT(t):
  r'\d+\.\d*|\.\d+'
  t.value = float(t.value)
  return t

def t_INT(t):
  r'\d+'
  t.value = int(t.value)
  return t

def t_error(t):
  print(f"Illegal character {t.value[0]}")
  t.lexer.skip(1)

__file__ = "Untitled.ipynb" # needed to run PLY inside notebook
lexer = lex.lex()

In [ ]:
def run_lexer(data):
    lexer.input(data)
    return [(tok.type, tok.value) for tok in lexer]

print(run_lexer("10+20"))
print(run_lexer("(10.23 + 20) * 293"))
print(run_lexer("10-20"))
print(run_lexer("10.23 + 20 * -293"))
print(run_lexer("10 - -20"))

## [`ply.yacc`](https://ply.readthedocs.io/en/latest/ply.html#yacc) basics

- Rules are given by special functions starting with `p_`
  - You can write a different function for each production, or merge them into a single function

- The grammar rules are given in the docstring in BNF

- The starting grammar symbol is the one defined first

- The terminals of the grammar are the token types of the lexer
   - They may not be constant: terminal `INT` is any integer number

- Each variable is assigned a *value* during parsing, defined by the semantic actions

- The semantic actions are executed in reduce steps
  - use the values of the sub-expressions (obtained through the appropriate index)
  - updates the value of variable being expanded

- For tokens (terminals) the value is that assigned during lexing

- `precedence` of operators can be given to disambiguate grammars

In [ ]:
import ply.yacc as yacc

import os
try: # avoid conflicts in temp files when running from notebook
    os.remove("parsetab.py")
except FileNotFoundError:
    pass

def p_expr_add_sub(p):
    '''expr : expr PLUS term
            | expr MINUS term'''
    if p[2] == '+':
        p[0] = p[1] + p[3]
    else:
        p[0] = p[1] - p[3]

def p_expr_term(p):
    'expr : term'
    p[0] = p[1]

def p_term_mul_div(p):
    '''term : term TIMES factor
            | term DIVIDE factor'''
    if p[2] == '*':
        p[0] = p[1] * p[3]
    else:
        p[0] = p[1] / p[3]

def p_term_factor(p):
    'term : factor'
    p[0] = p[1]

def p_factor_int(p):
    'factor : INT'
    p[0] = p[1]

def p_factor_float(p):
    'factor : FLOAT'
    p[0] = p[1]

def p_factor_parens(p):
    'factor : "(" expr ")"'
    p[0] = p[2]

def p_factor_uminus(p):
    'factor : MINUS factor'
    p[0] = -p[2]

def p_error(p):
    if p:
        print(f"Syntax error at '{p.value}'")
    else:
        print("Syntax error at EOF")

parser = yacc.yacc(write_tables=False, debug=True)

In [ ]:
print(parser.parse("(-3.4 * 30) + 20 + -30 * 100"))
print(parser.parse("(10 + 30 + 100) * (1 - 1)"))

# Exercise 1

Propositional logic formulas have the following syntax:

- Variables $A$, $B$, $C$, ...
- Conjunction $f1 ∧ f2$,
- Disjunction $f1 ∨ f2$,
- Negation $¬ f1$,
- Implication $f1 → f2$
- Equivalence $f1 ↔︎ f2$
- Constants true $⊤$ and false $⊥$

## Exercise 1.1

Write a lexer using `ply.lex` to tokenize propositional logic formulas.

In [ ]:
import ply.lex as lex

tokens = ("ID", "AND", "OR", "IMPLIES", "EQUIV", "NOT", "TRUE", "FALSE")

literals = "()"

t_AND = r"∧"
t_OR = r"∨"
t_IMPLIES = r"→"
t_EQUIV = r"↔︎"
t_NOT = r"¬"
t_FALSE = r"⊥"
t_TRUE = r"⊤"

t_ID = r"\w+"

t_ignore = ' \t\n'

def t_error(p):
  print("Erro", p)

__file__ = "Untitled.ipynb" # needed to run PLY inside notebook

lexer = lex.lex()

def run_lexer(data):
    """
    PLY lexer with doctests.

    >>> print([(tok.type, tok.value) for tok in run_lexer("A ∧ ¬B")])
    [('ID', 'A'), ('AND', '∧'), ('NOT', '¬'), ('ID', 'B')]
    >>> print([(tok.type, tok.value) for tok in run_lexer("A → (¬B ∨ C)")])
    [('ID', 'A'), ('IMPLIES', '→'), ('(', '('), ('NOT', '¬'), ('ID', 'B'), ('OR', '∨'), ('ID', 'C'), (')', ')')]
    >>> print([(tok.type, tok.value) for tok in run_lexer("(A → AB) ∧ ¬(¬B ∨ C)")])
    [('(', '('), ('ID', 'A'), ('IMPLIES', '→'), ('ID', 'AB'), (')', ')'), ('AND', '∧'), ('NOT', '¬'), ('(', '('), ('NOT', '¬'), ('ID', 'B'), ('OR', '∨'), ('ID', 'C'), (')', ')')]
    >>> print([(tok.type, tok.value) for tok in run_lexer("⊥ ∧ aux")])
    [('FALSE', '⊥'), ('AND', '∧'), ('ID', 'aux')]
    """

    lexer.input(data)
    return lexer

import doctest
doctest.run_docstring_examples(run_lexer, globals())

## Exercise 1.2

Write a parser using `ply.yacc` to parse and retrieve all propositional variables from a propositional formula.

In [ ]:
import ply.yacc as yacc

import os
try: # avoid conflicts in temp files when running from notebook
    os.remove("parsetab.py")
except FileNotFoundError:
    pass

def p_equivs(elems):
  """equiv : equiv EQUIV impl
           | impl"""
  if len(elems) == 4:
    elems[0] = elems[1] + elems[3]
  else:
    elems[0] = elems[1]

def p_implications(elems):
  """impl : impl IMPLIES disj
          | disj"""
  if len(elems) == 4:
    elems[0] = elems[1] + elems[3]
  else:
    elems[0] = elems[1]

def p_disj(elems):
  """disj : disj OR conj
          | conj"""
  if len(elems) == 4:
    elems[0] = elems[1] + elems[3]
  else:
    elems[0] = elems[1]

def p_conj(elems):
  """conj : conj AND neg
          | neg"""
  if len(elems) == 4:
    elems[0] = elems[1] + elems[3]
  else:
    elems[0] = elems[1]

def p_neg(elems):
  """neg : NOT term
         | '(' equiv ')'
         | term"""
  if len(elems) == 4:
    elems[0] = elems[2]
  elif len(elems) == 3:
    elems[0] = elems[2]
  else:
    elems[0] = elems[1]

def p_term(elems):
  """term : var
          | constant"""
  elems[0] = elems[1]

def p_constant(elems):
  """constant : FALSE
             | TRUE"""
  elems[0] = []

def p_var(elems):
  """var : ID"""
  elems[0] = [elems[1]]

def p_error(elems):
  print("Erro", elems)

def run_parser(data):
  """
  PLY parser with doctests.

  >>> print(run_parser("Chove ∧ ¬Frio"))
  {'Chove', 'Frio'}
  >>> print(run_parser("Dia ∨ ¬Dia"))
  {'Dia'}
  >>> print(run_parser("A ∧ ¬B ∨ ⊤"))
  {'A', 'B'}
  >>> print(run_parser("Reprovar → (¬Presenca ∨ Negativa)"))
  {'Presenca', 'Reprovar', 'Negativa'}
  >>> print(run_parser("A ∧ (X ∧ ¬Y) ∨ ⊤"))
  {'A', 'X', 'Y'}
  """
  return yacc.yacc(write_tables=False, debug=True).parse(data)

import doctest
doctest.run_docstring_examples(run_parser, globals())

## Exercise 1.3

Write a parser using `ply.yacc` to parse and evaluate the truth value of a propositional formula.

In [ ]:
import ply.yacc as yacc

import os
try: # avoid conflicts in temp files when running from notebook
    os.remove("parsetab.py")
except FileNotFoundError:
    pass

assignment = None

# define the PLY lexer
# ...

def run_parser(data, asg):
  """
  PLY parser with doctests.

  >>> print(run_parser("Frio ∧ ¬Chove", {"Frio":True, "Chove": True}))
  False
  >>> print(run_parser("Noite ∨ ¬Noite", {"Noite":True, "Dia": False}))
  True
  >>> print(run_parser("A ∧ ¬B ∨ ⊤", {"A":True, "B": True}))
  True
  >>> print(run_parser("Reprovar → (¬Presenca ∨ Negativa)", {"Presenca":True, "Negativa": False, "Reprovar": False}))
  True
  >>> print(run_parser("A ∧ (X ∧ ¬Y)", {"A":True, "X": False, "Y": True}))
  False
  """
  global assignment # not good: hack to run inside notbook
  assignment = asg
  return yacc.yacc(write_tables=False, debug=True).parse(data)

import doctest
doctest.run_docstring_examples(run_parser, globals())

# Exercise 2

Consider a language to specify monotonic numeric ranges: either always increasing or always decreasing. For instance, the following are valid ranges:
- `[3 ; 3.5] [5.6 ; 12] [-10 ; 2.5]`
- `[3 ; 2.5] [5 ; 1.2] [1 ; -0.8]`

## Exercise 2.1

Write a lexer using `ply.lex` to tokenize strings in this language.

In [ ]:
import ply.lex as lex
import math

import ply.lex as lex
import math

# define the PLY lexer
# ...

__file__ = "Untitled0.ipynb" # needed to run PLY inside notebook

lexer = lex.lex()

def run_lexer(data):
    """
    PLY lexer with doctests.

    >>> print([(tok.type, tok.value) for tok in run_lexer('[-1.5 ; +2] [3 ; +4.7]')])
    [('[', '['), ('NUM', -1.5), (';', ';'), ('NUM', 2.0), (']', ']'), ('[', '['), ('NUM', 3.0), (';', ';'), ('NUM', 4.7), (']', ']')]
    >>> print([(tok.type, tok.value) for tok in run_lexer("[3 ; 3.5] [5 ; 12] [48 ; 73.2]")])
    [('[', '['), ('NUM', 3.0), (';', ';'), ('NUM', 3.5), (']', ']'), ('[', '['), ('NUM', 5.0), (';', ';'), ('NUM', 12.0), (']', ']'), ('[', '['), ('NUM', 48.0), (';', ';'), ('NUM', 73.2), (']', ']')]
    >>> print([(tok.type, tok.value) for tok in run_lexer("[3 ; 3.5] [1 ; 12] [48 ; 73.2]")])
    [('[', '['), ('NUM', 3.0), (';', ';'), ('NUM', 3.5), (']', ']'), ('[', '['), ('NUM', 1.0), (';', ';'), ('NUM', 12.0), (']', ']'), ('[', '['), ('NUM', 48.0), (';', ';'), ('NUM', 73.2), (']', ']')]
    >>> print([(tok.type, tok.value) for tok in run_lexer("[3 ; 3.5] [2 ; 1] [0.5 ; -73.2]")])
    [('[', '['), ('NUM', 3.0), (';', ';'), ('NUM', 3.5), (']', ']'), ('[', '['), ('NUM', 2.0), (';', ';'), ('NUM', 1.0), (']', ']'), ('[', '['), ('NUM', 0.5), (';', ';'), ('NUM', -73.2), (']', ']')]
    """

    lexer.input(data)
    return lexer

import doctest
doctest.run_docstring_examples(run_lexer, globals())

## Exercise 2.2

Write a parser using `ply.yacc` to parse count the number of intervals defined and sum of all intervals.

In [ ]:
import ply.yacc as yacc

import os
try: # avoid conflicts in temp files when running from notebook
    os.remove("parsetab.py")
except FileNotFoundError:
    pass

# define the PLY parser
# ...

def run_parser(data):
  """
  PLY parser with doctests.

  >>> print(run_parser('[-1 ; 0] [0 ; 1] [2 ; 3] [3 ; 4]'))
  (4, 4.0)
  >>> print(run_parser('[-200 ; -123] [-100 ; 0] [-1.5 ; +2] [3 ; +4.7]'))
  (4, 182.2)
  >>> print(run_parser("[3 ; 3.5] [5 ; 12] [48 ; 73.2]"))
  (3, 32.7)
  >>> print(run_parser("[34 ; 36.5]"))
  (1, 2.5)
  >>> print(run_parser("[3 ; 3.5] [1 ; 12] [48 ; 73.2]"))
  (3, 36.7)
  >>> print(run_parser("[3 ; 3.5] [2 ; 1] [0.5 ; -73.2]"))
  (3, -74.2)
  """
  return yacc.yacc(write_tables=False, debug=True).parse(data)

import doctest
doctest.run_docstring_examples(run_parser, globals())

## Exercise 2.3

Write a parser to test whether the ranges are valid (always decreasing or always increasing).

In [ ]:
import ply.yacc as yacc

import os
try: # avoid conflicts in temp files when running from notebook
    os.remove("parsetab.py")
except FileNotFoundError:
    pass

# define the PLY parser
# ...

def run_parser(data):
  """
  PLY parser with doctests.

  >>> print(run_parser('[-1 ; 0] [0 ; 1] [2 ; 3] [3 ; 4]'))
  True
  >>> print(run_parser('[-200 ; -123] [-100 ; 0] [-1.5 ; +2] [3 ; +4.7]'))
  True
  >>> print(run_parser("[3 ; 3.5] [5 ; 12] [48 ; 13.2]"))
  False
  >>> print(run_parser("[34 ; 36.5]"))
  True
  >>> print(run_parser("[3 ; 3.5] [63 ; 12] [48 ; 73.2]"))
  False
  >>> print(run_parser("[3 ; 3.5] [2 ; 1] [0.5 ; -73.2]"))
  False
  """
  return yacc.yacc(write_tables=False, debug=True).parse(data)

import doctest
doctest.run_docstring_examples(run_parser, globals())

# Module [`Lark`](https://lark-parser.readthedocs.io/) basics

- In `Lark`, the grammar is purely syntactic, there are no semantic actions assigned

- It automatically generates an AST and provides methods to traverse it

- By decoupling the parsing and the transformation, it is more prone to multiple analysis

- No clear distinction between the lexer and the parser, defined in the same specification

- Richer range of parsing algorithms, supports more grammars

## `Lark` grammar

- Grammar is in extended EBNF
  - operators for repetition of elements `*`, optional elements `?`, ...

- Grammar starting symbol identified with `start`

- Terminal definition is at the same level as the non-terminals
  - Can use regex, identified by `/` delimiters
  - Some predefined terminals can be imported with instruction `%import`

- Unnamed literals in grammar rules ignored in output
  - `?` also marks others variables to be ignored

- Rules can be assigned names to ease writing transformations

- Precedence and associativity rules are declared with instructions `%left` and `%right`

- `Lark` also supports parsing algorithms that return multiple trees when ambiguous

In [ ]:
from lark import Lark, Transformer, v_args

grammar = """
start: expr

expr: expr ADD term
    | expr SUB term
    | term

term: term MUL factor
    | term DIV factor
    | factor

factor: NUMBER
      | SUB factor
      | "(" expr ")"

ADD: "+"
SUB: "-"
MUL: "*"
DIV: "/"

NUMBER: /[0-9]+(\\.[0-9]+)?/

%ignore /[ \t\f]+/
"""

parser = Lark(grammar)

In [ ]:
print(parser.parse("(-3.4 * 30) + 20 + -30 * 100").pretty())
print(parser.parse("(10 + 30 + 100) * (1 - 1)").pretty())

## `Lark` transformers

- `Lark` automatically creates an AST for the parsed strings

- Provides helper classes that ease the traversal of these ASTs

- `Transformer` is one such class, performs a bottom-up traversal
  - It provides a methods for each grammar rule, which should be overwritten
  - The argument is a list of values resulting from processing the children nodes

- In the grammar, aliases for productions can be defined with `->`

In [ ]:

class EvalTransformer(Transformer):
    def start(self, items):
        return items[0]

    def expr(self, items):
        if len(items) == 1:
          return items[0]
        left, op, right = items
        if op.type == 'ADD':
            return left + right
        elif op.type == 'SUB':
            return left - right

    def term(self, items):
        if len(items) == 1:
          return items[0]
        left, op, right = items
        if op.type == 'MUL':
            return left * right
        elif op.type == 'DIV':
            return left / right

    def factor(self, items):
        if len(items) == 1:
            return items[0]
        elif len(items) == 2:
            return -items[1]
        return items[1]

    def NUMBER(self, token):
        return float(token)

In [ ]:
print(EvalTransformer().transform(parser.parse("(-3.4 * 30) + 20 + -30 * 100")))
print(EvalTransformer().transform(parser.parse("(10 + 30 + 100) * (1 - 1)")))

# Exercise 3

Recall the language for propositional formulas from Exercise 1.

## Exercise 3.1

Write a `lark` grammar to obtain the AST for propositional formulas.

In [ ]:
from lark import Lark, Transformer, v_args

def run_parser(data):
  """
  Lark parser with doctests.

  >>> run_parser("A ∧ ¬B")
  Tree(Token('RULE', 'start'), [Tree(Token('RULE', 'expr'), [Tree(Token('RULE', 'expr'), [Token('ID', 'A')]), Token('AND', '∧'), Tree(Token('RULE', 'expr'), [Token('NOT', '¬'), Tree(Token('RULE', 'expr'), [Token('ID', 'B')])])])])

  >>> run_parser("A ∧ ¬B")
  Tree(Token('RULE', 'start'), [Tree(Token('RULE', 'expr'), [Tree(Token('RULE', 'expr'), [Token('ID', 'A')]), Token('AND', '∧'), Tree(Token('RULE', 'expr'), [Token('NOT', '¬'), Tree(Token('RULE', 'expr'), [Token('ID', 'B')])])])])

  >>> run_parser("A ∧ ¬B ∨ ⊤")
  Tree(Token('RULE', 'start'), [Tree(Token('RULE', 'expr'), [Tree(Token('RULE', 'expr'), [Token('ID', 'A')]), Token('AND', '∧'), Tree(Token('RULE', 'expr'), [Tree(Token('RULE', 'expr'), [Token('NOT', '¬'), Tree(Token('RULE', 'expr'), [Token('ID', 'B')])]), Token('OR', '∨'), Tree(Token('RULE', 'expr'), [Token('FALSE', '⊤')])])])])

  >>> run_parser("A ∧ ¬B ∨ ⊤")
  Tree(Token('RULE', 'start'), [Tree(Token('RULE', 'expr'), [Tree(Token('RULE', 'expr'), [Token('ID', 'A')]), Token('AND', '∧'), Tree(Token('RULE', 'expr'), [Tree(Token('RULE', 'expr'), [Token('NOT', '¬'), Tree(Token('RULE', 'expr'), [Token('ID', 'B')])]), Token('OR', '∨'), Tree(Token('RULE', 'expr'), [Token('FALSE', '⊤')])])])])

  """
  grammar = r"""
  start : expr
  
  expr : expr EQUIV impl
          | impl

  impl: impl IMPLIES disj
      | disj
 
  disj: disj DIS conj
      | conj

  conj: conj CON neg
      | neg

  neg: NEG term
     | term

  term: var | constant | paren

  paren: "(" expr ")"
  var: ID
  constant: TRUE | FALSE

  CON: "∧"
  IMPLIES: "→"
  EQUIV: "↔︎"
  DIS: "∨"
  NEG: "¬"
  TRUE: "⊤"
  FALSE: "⊥"

  ID : /\w+/

  %ignore /[ \t\f]+/

  """

  parser = Lark(grammar)
  return parser.parse(data)

import doctest
doctest.run_docstring_examples(run_parser, globals())

## Exercise 3.2

Write a `lark` transformer to retrieve all propositional variables of a propositional formula.

In [ ]:
def vars(data):
  """
  Lark variable collector with doctests.

  >>> print(vars("A ∧ ¬B"))
  {'A', 'B'}
  >>> print(vars("A ∨ ¬A"))
  {'A'}
  >>> print(vars("A ∧ ¬B ∨ ⊤"))
  {'A', 'B'}
  >>> print(vars("A → (¬B ∨ C)"))
  {'C', 'A', 'B'}
  >>> print(vars("A ∧ (X ∧ ¬Y) ∨ ⊤"))
  {'X', 'A', 'Y'}
  """
  class EvalTransformer(Transformer):
      def start(self, items):
        return items[0]

      def expr(self, items):
        if len(items) == 1:
          return items[0]
        else:
          return items[0] | items[2]
      
      def impl(self, items):
        if len(items) == 1:
          return items[0]
        else:
          return items[0] | items[2]
      
      def disj(self, items):
        if len(items) == 1:
          return items[0]
        else:
          return items[0] | items[2]
        
      def conj(self, items):
        if len(items) == 1:
          return items[0]
        else:
          return items[0] | items[2]
      
      def neg(self, items):
        if len(items) == 1:
          return items[0]
        else:
          return items[1]
      
      def term(self, items):
        return items[0]
      
      def constant(self, items):
          return set()
      
      def paren(self, items):
          return items[0]
      
      def var(self, items):
          return {items[0].value}

  return EvalTransformer().transform(run_parser(data))

import doctest
doctest.run_docstring_examples(vars, globals())

## Exercise 3.3

Write a `lark` transformer to evaluate the truth value of a propositional formula.

In [ ]:
def eval(data, assignment):
  """
  Lark evaluator with doctests.

  >>> print(eval("A ∧ ¬B", {"A":True, "B": True}))
  False
  >>> print(eval("A ∨ ¬A", {"A":True, "B": False}))
  True
  >>> print(eval("A ∧ ¬B ∨ ⊤", {"A":True, "B": True}))
  True
  >>> print(eval("A → (¬B ∨ C)", {"A":True, "B": False, "C": False}))
  True
  >>> print(eval("A ∧ (X ∧ ¬Y)", {"A":True, "X": False, "Y": True}))
  False
  """
  class EvalTransformer(Transformer):
      def start(self, items):
        return items[0]

      def expr(self, items):
        if len(items) == 1:
          return items[0]
        else:
          return items[0] == items[2]
      
      def impl(self, items):
        if len(items) == 1:
          return items[0]
        else:
          return not items[0] or items[2]
      
      def disj(self, items):
        if len(items) == 1:
          return items[0]
        else:
          return items[0] or items[2]
        
      def conj(self, items):
        if len(items) == 1:
          return items[0]
        else:
          return items[0] and items[2]
      
      def neg(self, items):
        if len(items) == 1:
          return items[0]
        else:
          return not items[1]
      
      def term(self, items):
        return items[0]
      
      def constant(self, items):
        if items[0].type == "FALSE":
          return False
        else:
            return True
      
      def paren(self, items):
          return items[0]
      
      def var(self, items):
          return assignment[items[0].value]
  
  return EvalTransformer().transform(run_parser(data))

import doctest
doctest.run_docstring_examples(eval, globals())

# Exercise 4

Recall the language for intervals from Exercise 2.

## Exercise 4.1

Write a `lark` grammar to obtain the AST for interval ranges.

In [62]:
from lark import Lark, Transformer, v_args

def run_parser(data):
  """
  Lark parser with doctests.

  >>> run_parser('[-1.5 ; +2] [3 ; +4.7]')
  Tree(Token('RULE', 'start'), [Tree(Token('RULE', 'intervals'), [Tree(Token('RULE', 'intervals'), [Tree(Token('RULE', 'intervals'), []), Tree(Token('RULE', 'interval'), [Tree(Token('RULE', 'number'), [Token('FLOAT', '-1.5')]), Tree(Token('RULE', 'number'), [Token('INT', '+2')])])]), Tree(Token('RULE', 'interval'), [Tree(Token('RULE', 'number'), [Token('INT', '3')]), Tree(Token('RULE', 'number'), [Token('FLOAT', '+4.7')])])])])

  >>> run_parser("[3 ; 3.5] [5 ; 12] [48 ; 73.2]")
  Tree(Token('RULE', 'start'), [Tree(Token('RULE', 'intervals'), [Tree(Token('RULE', 'intervals'), [Tree(Token('RULE', 'intervals'), [Tree(Token('RULE', 'intervals'), []), Tree(Token('RULE', 'interval'), [Tree(Token('RULE', 'number'), [Token('INT', '3')]), Tree(Token('RULE', 'number'), [Token('FLOAT', '3.5')])])]), Tree(Token('RULE', 'interval'), [Tree(Token('RULE', 'number'), [Token('INT', '5')]), Tree(Token('RULE', 'number'), [Token('INT', '12')])])]), Tree(Token('RULE', 'interval'), [Tree(Token('RULE', 'number'), [Token('INT', '48')]), Tree(Token('RULE', 'number'), [Token('FLOAT', '73.2')])])])])

  >>> run_parser("[3 ; 3.5] [1 ; 12] [48 ; 73.2]")
  Tree(Token('RULE', 'start'), [Tree(Token('RULE', 'intervals'), [Tree(Token('RULE', 'intervals'), [Tree(Token('RULE', 'intervals'), [Tree(Token('RULE', 'intervals'), []), Tree(Token('RULE', 'interval'), [Tree(Token('RULE', 'number'), [Token('INT', '3')]), Tree(Token('RULE', 'number'), [Token('FLOAT', '3.5')])])]), Tree(Token('RULE', 'interval'), [Tree(Token('RULE', 'number'), [Token('INT', '1')]), Tree(Token('RULE', 'number'), [Token('INT', '12')])])]), Tree(Token('RULE', 'interval'), [Tree(Token('RULE', 'number'), [Token('INT', '48')]), Tree(Token('RULE', 'number'), [Token('FLOAT', '73.2')])])])])

  >>> run_parser("[3 ; 3.5] [2 ; 1] [0.5 ; -73.2]")
  Tree(Token('RULE', 'start'), [Tree(Token('RULE', 'intervals'), [Tree(Token('RULE', 'intervals'), [Tree(Token('RULE', 'intervals'), [Tree(Token('RULE', 'intervals'), []), Tree(Token('RULE', 'interval'), [Tree(Token('RULE', 'number'), [Token('INT', '3')]), Tree(Token('RULE', 'number'), [Token('FLOAT', '3.5')])])]), Tree(Token('RULE', 'interval'), [Tree(Token('RULE', 'number'), [Token('INT', '2')]), Tree(Token('RULE', 'number'), [Token('INT', '1')])])]), Tree(Token('RULE', 'interval'), [Tree(Token('RULE', 'number'), [Token('FLOAT', '0.5')]), Tree(Token('RULE', 'number'), [Token('FLOAT', '-73.2')])])])])
  """

  grammar = r"""
  start : intervals
  
  intervals : interval intervals
          |

  interval: "[" number ";" number "]"

  number : INT | FLOAT

  FLOAT : /[+-]?\d+\.\d+/
  INT   : /[+-]?\d+/

  %ignore /[ \t\f]+/

  """

  parser = Lark(grammar)
  return parser.parse(data)

import doctest
doctest.run_docstring_examples(run_parser, globals())

**********************************************************************
File "__main__", line 7, in NoName
Failed example:
    run_parser('[-1.5 ; +2] [3 ; +4.7]')
Expected:
    Tree(Token('RULE', 'start'), [Tree(Token('RULE', 'intervals'), [Tree(Token('RULE', 'intervals'), [Tree(Token('RULE', 'intervals'), []), Tree(Token('RULE', 'interval'), [Tree(Token('RULE', 'number'), [Token('FLOAT', '-1.5')]), Tree(Token('RULE', 'number'), [Token('INT', '+2')])])]), Tree(Token('RULE', 'interval'), [Tree(Token('RULE', 'number'), [Token('INT', '3')]), Tree(Token('RULE', 'number'), [Token('FLOAT', '+4.7')])])])])
Got:
    Tree(Token('RULE', 'start'), [Tree(Token('RULE', 'intervals'), [Tree(Token('RULE', 'interval'), [Tree(Token('RULE', 'number'), [Token('FLOAT', '-1.5')]), Tree(Token('RULE', 'number'), [Token('INT', '+2')])]), Tree(Token('RULE', 'intervals'), [Tree(Token('RULE', 'interval'), [Tree(Token('RULE', 'number'), [Token('INT', '3')]), Tree(Token('RULE', 'number'), [Token('FLOAT', '+4.7')])]

## Exercise 4.2

Write a `lark` transformer to obtain obtain the number of intervals and overall range.

In [63]:
def stats(data):
  """
  Lark statististics collector with doctests.

  >>> print(stats('[-1 ; 0] [0 ; 1] [2 ; 3] [3 ; 4]'))
  (4, 4.0)
  >>> print(stats('[-200 ; -123] [-100 ; 0] [-1.5 ; +2] [3 ; +4.7]'))
  (4, 182.2)
  >>> print(stats("[3 ; 3.5] [5 ; 12] [48 ; 73.2]"))
  (3, 32.7)
  >>> print(stats("[34 ; 36.5]"))
  (1, 2.5)
  >>> print(stats("[3 ; 3.5] [1 ; 12] [48 ; 73.2]"))
  (3, 36.7)
  >>> print(stats("[3 ; 3.5] [2 ; 1] [0.5 ; -73.2]"))
  (3, -74.2)
  """
  class StatsTransformer(Transformer):

      def start(self, items):
        return items[0]

      def intervals(self, items):
          if len(items) == 0:
              return (0, 0.0)

          if len(items) == 2:
              this_total = items[0]

              sub_count, sub_total = items[1]

              return (sub_count + 1, sub_total + this_total)
          else:
              return (1, items[0])
      
      def interval(self, items):
        a, b = items
        return b - a
    
      def number(self, items):
        return float(items[0])
        
  return StatsTransformer().transform(run_parser(data))

import doctest
doctest.run_docstring_examples(stats, globals())

## Exercise 4.3

Write a `lark` transformer to validate of intervals (always increasing or always decreasing)

In [79]:
def validate(data):
  """
  Lark interval validator with doctests.

  >>> print(validate('[-1 ; 0] [0 ; 1] [2 ; 3] [3 ; 4]'))
  True
  >>> print(validate('[-200 ; -123] [-100 ; 0] [-1.5 ; +2] [3 ; +4.7]'))
  True
  >>> print(validate("[3 ; 3.5] [5 ; 12] [48 ; 13.2]"))
  False
  >>> print(validate("[34 ; 36.5]"))
  True
  >>> print(validate("[3 ; 3.5] [63 ; 12] [48 ; 73.2]"))
  False
  >>> print(validate("[3 ; 3.5] [2 ; 1] [0.5 ; -73.2]"))
  False
  >>> print(validate("[3.5 ; 3] [2 ; 1] [0.5 ; -73.2]"))
  True
  """
  class ValidationTransformer(Transformer):
      def start(self, items):
        return items[0][0]

      def intervals(self, items):
          if len(items) == 0:
             return (True, None)
          if len(items) == 2:
              this_valid = items[0]
              validation, sub_valid = items[1]

              if validation:
                if sub_valid == this_valid or sub_valid is None:
                    return (True, this_valid)
              return (False, sub_valid)
          else:
              return (True, items[0])
      
      def interval(self, items):
        a, b = items
        if a <= b:
           return "up"
        return "down"
    
      def number(self, items):
        return float(items[0])
      
  return ValidationTransformer().transform(run_parser(data))

import doctest
doctest.run_docstring_examples(validate, globals())

# Exercise 5

Consider a tiny functional language that supports:

- Integer and Boolean literals
- Binary operators (`+`, `*`, `<`, `and`, `or`)
- Conditional expressions (`if ... then ... else ...`)

For instance, the following is a valid expression in this language:
```
if 2 > 3 or 4 < 9 then 10 else -1*10
```

## Exercise 5.1

Write a `lark` grammar to obtain the AST for this tiny language.

In [ ]:
from lark import Lark, Transformer, v_args

def run_parser(data):
  """
  Lark parser with doctests.

  >>> run_parser('if 2 > 3 or 4 < 9 then 10 else -1*10')
  Tree(Token('RULE', 'start'), [Tree(Token('RULE', 'expr'), [Tree(Token('RULE', 'binexpr'), [Tree(Token('RULE', 'expr'), [Tree(Token('RULE', 'ifthenelse'), [Tree(Token('RULE', 'expr'), [Tree(Token('RULE', 'binexpr'), [Tree(Token('RULE', 'expr'), [Tree(Token('RULE', 'binexpr'), [Tree(Token('RULE', 'expr'), [Tree(Token('RULE', 'constant'), [Token('INT', '2')])]), Tree(Token('RULE', 'expr'), [Tree(Token('RULE', 'binexpr'), [Tree(Token('RULE', 'expr'), [Tree(Token('RULE', 'constant'), [Token('INT', '3')])]), Tree(Token('RULE', 'expr'), [Tree(Token('RULE', 'constant'), [Token('INT', '4')])])])])])]), Tree(Token('RULE', 'expr'), [Tree(Token('RULE', 'constant'), [Token('INT', '9')])])])]), Tree(Token('RULE', 'expr'), [Tree(Token('RULE', 'constant'), [Token('INT', '10')])]), Tree(Token('RULE', 'expr'), [Tree(Token('RULE', 'constant'), [Token('INT', '-1')])])])]), Tree(Token('RULE', 'expr'), [Tree(Token('RULE', 'constant'), [Token('INT', '10')])])])])])

  >>> run_parser('if 2 > 3 or 4 then 10 else -1*10')
  Tree(Token('RULE', 'start'), [Tree(Token('RULE', 'expr'), [Tree(Token('RULE', 'binexpr'), [Tree(Token('RULE', 'expr'), [Tree(Token('RULE', 'ifthenelse'), [Tree(Token('RULE', 'expr'), [Tree(Token('RULE', 'binexpr'), [Tree(Token('RULE', 'expr'), [Tree(Token('RULE', 'constant'), [Token('INT', '2')])]), Tree(Token('RULE', 'expr'), [Tree(Token('RULE', 'binexpr'), [Tree(Token('RULE', 'expr'), [Tree(Token('RULE', 'constant'), [Token('INT', '3')])]), Tree(Token('RULE', 'expr'), [Tree(Token('RULE', 'constant'), [Token('INT', '4')])])])])])]), Tree(Token('RULE', 'expr'), [Tree(Token('RULE', 'constant'), [Token('INT', '10')])]), Tree(Token('RULE', 'expr'), [Tree(Token('RULE', 'constant'), [Token('INT', '-1')])])])]), Tree(Token('RULE', 'expr'), [Tree(Token('RULE', 'constant'), [Token('INT', '10')])])])])])

  >>> run_parser('if 2 > 3 or 4 then 10 else -1*(10 or 20)')
  Tree(Token('RULE', 'start'), [Tree(Token('RULE', 'expr'), [Tree(Token('RULE', 'binexpr'), [Tree(Token('RULE', 'expr'), [Tree(Token('RULE', 'ifthenelse'), [Tree(Token('RULE', 'expr'), [Tree(Token('RULE', 'binexpr'), [Tree(Token('RULE', 'expr'), [Tree(Token('RULE', 'constant'), [Token('INT', '2')])]), Tree(Token('RULE', 'expr'), [Tree(Token('RULE', 'binexpr'), [Tree(Token('RULE', 'expr'), [Tree(Token('RULE', 'constant'), [Token('INT', '3')])]), Tree(Token('RULE', 'expr'), [Tree(Token('RULE', 'constant'), [Token('INT', '4')])])])])])]), Tree(Token('RULE', 'expr'), [Tree(Token('RULE', 'constant'), [Token('INT', '10')])]), Tree(Token('RULE', 'expr'), [Tree(Token('RULE', 'constant'), [Token('INT', '-1')])])])]), Tree(Token('RULE', 'expr'), [Tree(Token('RULE', 'expr'), [Tree(Token('RULE', 'binexpr'), [Tree(Token('RULE', 'expr'), [Tree(Token('RULE', 'constant'), [Token('INT', '10')])]), Tree(Token('RULE', 'expr'), [Tree(Token('RULE', 'constant'), [Token('INT', '20')])])])])])])])])

  >>> run_parser('if 2 > 3 then 10 else True')
  Tree(Token('RULE', 'start'), [Tree(Token('RULE', 'expr'), [Tree(Token('RULE', 'ifthenelse'), [Tree(Token('RULE', 'expr'), [Tree(Token('RULE', 'binexpr'), [Tree(Token('RULE', 'expr'), [Tree(Token('RULE', 'constant'), [Token('INT', '2')])]), Tree(Token('RULE', 'expr'), [Tree(Token('RULE', 'constant'), [Token('INT', '3')])])])]), Tree(Token('RULE', 'expr'), [Tree(Token('RULE', 'constant'), [Token('INT', '10')])]), Tree(Token('RULE', 'expr'), [Tree(Token('RULE', 'constant'), [Token('BOOL', 'True')])])])])])
  """
  grammar = None # define the Lark grammar

  parser = Lark(grammar)
  return parser.parse(data)

import doctest
doctest.run_docstring_examples(run_parser, globals())

## Exercise 5.2

Write a `lark` transformer to type check this language.

For instance, the following expression type checks:
```
if 2 > 3 or 4 < 9 then 10 else -1*10
```

But the following (syntactically correct) should throw a type error:

```
if 2 > 3 or 4 then 10 else -1*10
```
```
if 2 > 3 or 4 then 10 else -1*(10 or 20)
```
```
if 2 > 3 then 10 else True
```


In [ ]:
def validate(data):
  """
  Lark type checker with doctests.

  >>> check('if 2 > 3 or 4 < 9 then 10 else -1*10')
  True

  >>> check('if 2 > 3 or 4 then 10 else -1*10')
  False

  >>> check('if 2 > 3 or 4 then 10 else -1*(10 or 20)')
  False

  >>> check('if 2 > 3 then 10 else True')
  False
  """
  class ValidationTransformer(Transformer):

      # define the Lark transformer
      # ...

  return ValidationTransformer().transform(run_parser(data))

import doctest
doctest.run_docstring_examples(check, globals())

-- Nuno Macedo